In [ ]:
!pip install simpy

# 시뮬레이션 상황

# <IB>
##(오전) 15초에 1건 주문 : 주문량 1시간에 240건
##(오후) 30초에 1건 주문 : 주문량 1시간에 120건

# 주목할 점
##1. 현실성 : 점심/휴식 시간에 인력 감소 반영
##2. 공정별 분석 가능성 : 각 단계별 큐 및 소요 시간 추적
##3. 시뮬레이션 기반 의사결정
- 인력 추가 시 효과 예측 가능(코드상으로 인력투입이 Lane추가)
- 공정 병목 완화 전략 수립 가능

## 이 시뮬레이션은 단순한 공정 흐름 재현이 아니라, 운영 시간 내에서의 리소스 활용, 병목 현상, 처리 효율성 등을 정량적으로 분석하는 데 목적 둠

### IB :2, PICK :2, PACK :2, HUB:2

In [ ]:
import simpy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 시드 고정
random.seed(42)

# 작업 처리 시간 (초)
PROCESS_TIMES = {
    'IB': 20,
    'Picking': 60,
    'Packing': 10,
    'HUB': 20
}

# 시뮬레이션 설정
WORK_TIME = 9 * 60 * 60   # 9시간 = 32400초
ORDER_INTERVAL_STD_MORNING = 2
ORDER_INTERVAL_STD_AFTERNOON = 3

LUNCH_START = 2 * 60 * 60  # 7200초=11시
LUNCH_END = 4 * 60 * 60    # 14400초=1시
BREAK_PERIODS = [
    (5 * 60 * 60, 5 * 60 * 60 + 45 * 60),     # 14:00 ~ 14:45 (오후 2:00 ~ 2:45)
    (7 * 60 * 60, 7 * 60 * 60 + 45 * 60),     # 16:00 ~ 16:45 (오후 4:00 ~ 4:45)
]

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
        }

    def is_reduced_capacity(self):                # 점심 휴식 T/F 판단
        now = self.env.now
        if LUNCH_START <= now <= LUNCH_END:
            return True
        for start, end in BREAK_PERIODS:
            if start <= now <= end:
                return True
        return False

    def get_resource(self, stage):                 # 각 공정의 Resource(1or2) 반환
        return self.resources[stage][1] if self.is_reduced_capacity() else self.resources[stage][2]

    def process(self, stage):                       # "지금 피킹 처리 시간~"
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):        # IB->HUB까지 순서대로 처리되는 전체 과정
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")

    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))

    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = (
            max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING))  # 오전 주문 간격 (정규분포)
            if env.now < 4 * 60 * 60
            else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))  # 오후 주문 간격
        )
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 시뮬레이션 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []
env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()

# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:  # 앞 100줄만 출력
    print(log)

# 결과 분석
plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (High Load Scenario)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n총 주문 수: {len(wait_times)}")


hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")


print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

# 공정별 큐 시각화
plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 간트 차트
stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50

for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)

plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()

▶ 시뮬레이션 결과를 바탕으로 주문 처리 시간의 분포와 각 공정의 대기열 변화 추이를 분석

▶시스템의 병목 구간과 성능 저하 구간 파악

## 그래프1
1. 왼쪽 (Peak)

처리 시간이 0~1000초 사이인 주문이 많다

→ 병목이 없고 빠르게 처리된 주문

주로 근무 초반이나 리소스가 충분한 시점

2. 우측으로 갈수록 긴 꼬리 (Long tail)

최대 16,000초 이상 걸린 주문도 존재

→ 시스템에 심각한 지연이 발생한 주문

병목 현상, 큐 대기, 점심/휴식시간 리소스 감소로 인해 발생

3. 중간 구간의 계단식 패턴

다양한 지연 조건에 따라 처리 시간이 다양하게 분포

전체적으로는 비대칭적 분포 (오른쪽 꼬리가 김 → 대기 누적 효과)

## 그래프 2
X축: 시뮬레이션 시간 (초)

Y축: 각 공정의 대기열 길이

1. Picking 공정의 심각한 적체

대기열 길이가 200건을 넘는 병목이 반복 발생

작업자 수가 감소하는 점심 및 휴식 시간에 급격히 증가

# IB :2, PICK :2, PACK :2, HUB:2 에 휴식을 15분으로 줄임

In [ ]:
import simpy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 시드 고정
random.seed(42)

# 작업 처리 시간 (초)
PROCESS_TIMES = {
    'IB': 20,
    'Picking': 60,
    'Packing': 10,
    'HUB': 20
}

# 시뮬레이션 설정
WORK_TIME = 9 * 60 * 60   # 9시간 = 32400초
ORDER_INTERVAL_STD_MORNING = 2
ORDER_INTERVAL_STD_AFTERNOON = 3

LUNCH_START = 2 * 60 * 60  # 7200초=11시
LUNCH_END = 4 * 60 * 60    # 14400초=1시
BREAK_PERIODS = [
    (5 * 60 * 60, 5 * 60 * 60 + 15 * 60),  # 14:00 ~ 14:15
    (7 * 60 * 60, 7 * 60 * 60 + 15 * 60),  # 16:00 ~ 16:15
]

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
        }

    def is_reduced_capacity(self):
        now = self.env.now
        if LUNCH_START <= now <= LUNCH_END:
            return True
        for start, end in BREAK_PERIODS:
            if start <= now <= end:
                return True
        return False

    def get_resource(self, stage):
        return self.resources[stage][1] if self.is_reduced_capacity() else self.resources[stage][2]

    def process(self, stage):
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")

    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))

    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = (
            max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING))  # 오전 주문 간격
            if env.now < 4 * 60 * 60
            else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))  # 오후 주문 간격
        )
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 시뮬레이션 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []

env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()

# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:  # 앞 100줄만 출력
    print(log)

# 결과 분석
plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (High Load Scenario)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n총 주문 수: {len(wait_times)}")
hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")
print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

# 공정별 큐 시각화
plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 간트 차트
stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50

for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)

plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()

▶ 휴식시간 감소로 오후 처리 속도는 개선되었으나,

→ 처리량 증가가 발생해 여전히 queue는 유지됨



# 인력투입(하프타임 PICK 1명 추가)

In [ ]:
import simpy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 시드 고정
random.seed(42)

HALF_TIME_START = 0
HALF_TIME_END = 4 * 60 * 60  # 14400초 = 13:00


# 작업 처리 시간 (초)
PROCESS_TIMES = {
    'IB': 20,
    'Picking': 60,
    'Packing': 10,
    'HUB': 20
}

# 시뮬레이션 설정
WORK_TIME = 9 * 60 * 60
ORDER_INTERVAL_STD_MORNING = 2
ORDER_INTERVAL_STD_AFTERNOON = 3

LUNCH_START = 2 * 60 * 60
LUNCH_END = 4 * 60 * 60
BREAK_PERIODS = [
    (5 * 60 * 60, 5 * 60 * 60 + 15 * 60),  # 14:00 ~ 14:15
    (7 * 60 * 60, 7 * 60 * 60 + 15 * 60),  # 16:00 ~ 16:15
]

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3),  # 하프타임 인력 포함 시 최대 3명
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
        }

    def is_reduced_capacity(self):
        now = self.env.now
        if LUNCH_START <= now <= LUNCH_END:
            return True
        for start, end in BREAK_PERIODS:
            if start <= now <= end:
                return True
        return False

    def get_resource(self, stage):
        now = self.env.now
        if stage == 'Picking':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][3]  ## 하프타임 포함 3명
        if self.is_reduced_capacity():
            return self.resources[stage][1]
        else:
            return self.resources[stage][2]


    def process(self, stage):
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")

    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))

    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = (
            max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING))  # 오전 주문 간격
            if env.now < 4 * 60 * 60
            else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))  # 오후 주문 간격
        )
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 시뮬레이션 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []

env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()

# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:  # 앞 100줄만 출력
    print(log)

# 결과 분석
plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (High Load Scenario)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n총 주문 수: {len(wait_times)}")
hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")
print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

# 공정별 큐 시각화
plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.ylim(0, 250)
plt.yticks(range(0, 251, 50))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 간트 차트
stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50

for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)

plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()

# 입고 하프1명 & 피킹 하프 1명 추가

In [ ]:
HALF_TIME_START = 0
HALF_TIME_END = 4 * 60 * 60  # 14400초 = 13:00

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3)  # 하프타임용 추가 인력 포함
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3)  # 하프타임 인력 포함
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
        }

    def is_reduced_capacity(self):
        now = self.env.now
        if LUNCH_START <= now <= LUNCH_END:
            return True
        for start, end in BREAK_PERIODS:
            if start <= now <= end:
                return True
        return False

    def get_resource(self, stage):
      now = self.env.now
      # 하프타임일 경우 IB 또는 Picking에 추가 인력 투입
      if stage in ['IB', 'Picking'] and HALF_TIME_START <= now < HALF_TIME_END:
        return self.resources[stage][3]
      return self.resources[stage][1] if self.is_reduced_capacity() else self.resources[stage][2]


    def process(self, stage):
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")

    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))

    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = (
            max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING))  # 오전 주문 간격
            if env.now < 4 * 60 * 60
            else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))  # 오후 주문 간격
        )
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 시뮬레이션 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []

env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()

# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:  # 앞 100줄만 출력
    print(log)

# 결과 분석
plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (High Load Scenario)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n총 주문 수: {len(wait_times)}")
hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")
print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

# 공정별 큐 시각화
plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 간트 차트
stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50

for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)

plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()


### 1. 그래프 1

▶ 0~500초 구간에 450건? 이상 몰림

▶ 이전엔 10,000초 이상 걸리는 주문도 있었지만(인력2222일때), 지금은 5000초 이내로 대부분 처리

▶ 극단적인 지연 사라짐

### 2. 그래프2

▶ Picking 공정은 오전에 약간의 대기열이 생기지만, 점심 이후 즉시 해소되는 점에서 인력 투입이 매우 효과적

▶ 특히 IB 공정은 하프타임 인력 투입으로 병목이 거의 사라졌고, 전체 시스템 흐름이 크게 개선됨

# 입고 하프1, 피킹 하프1&풀1

In [ ]:
# 최종 요구 사항 반영한 전체 코드

import simpy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 시드 고정
random.seed(42)

# 작업 처리 시간 (초)
PROCESS_TIMES = {
    'IB': 20,
    'Picking': 60,
    'Packing': 10,
    'HUB': 20
}

# 시뮬레이션 설정
WORK_TIME = 9 * 60 * 60
ORDER_INTERVAL_STD_MORNING = 2
ORDER_INTERVAL_STD_AFTERNOON = 3

LUNCH_START = 2 * 60 * 60
LUNCH_END = 4 * 60 * 60
BREAK_PERIODS = [
    (5 * 60 * 60, 5 * 60 * 60 + 15 * 60),  # 14:00 ~ 14:15
    (7 * 60 * 60, 7 * 60 * 60 + 15 * 60),  # 16:00 ~ 16:15
]
HALF_TIME_START = 0
HALF_TIME_END = 4 * 60 * 60  # 13:00 (14400초)

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3)  # 기본 2 + 하프타임 1
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3),  # 기본 2 + 풀타임 1
                4: simpy.Resource(env, capacity=4)   # 기본 2 + 풀타임 1 + 하프타임 1
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
        }


    def is_reduced_capacity(self):
        now = self.env.now
        return LUNCH_START <= now <= LUNCH_END or any(start <= now <= end for start, end in BREAK_PERIODS)

    def get_resource(self, stage):
        now = self.env.now
        if stage == 'IB':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][3]  # 2명 + 하프타임 1명
        elif stage == 'Picking':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][4]  # 2명 + 풀타임 1 + 하프타임 1
            else:
                return self.resources[stage][3]  # 2명 + 풀타임 1
        return self.resources[stage][1] if self.is_reduced_capacity() else self.resources[stage][2]

    def process(self, stage):
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")
    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))
    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING)) if env.now < 4 * 60 * 60 else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 시뮬레이션 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []

env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()

# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:
    print(log)

# 결과 분석 시각화
plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (IB+Picking with Half-Time Support)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n총 주문 수: {len(wait_times)}")
hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")
print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

# 공정별 큐 시각화
plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.ylim(0, 250)
plt.yticks(range(0, 251, 50))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 간트 차트
stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50
for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)
plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()


###1. 그래프 1


###2. 그래프 2

▶약 9,000초 이후부터 급상승 → 14,400초(점심 전)에서 정점(120건)

▶즉, 앞 공정이 워낙 잘 돌아가니 뒤쪽 허브 공정이 감당 못함

▶전체 처리 속도는 매우 빠르지만, HUB 용량이 새 병목

좋은거

▶IB, Picking 공정은 병목 없음(오전 내내 선이 낮게 유지 → 인력 구성 최적)

---
# 여기서 HUB의 병목을 완화하기위해, 주문 우선순위 처리, 주문 분산, 작업 그케일링, PACK과 HUB사이 버퍼 도입 등 시도하였지만 결과는 미미했음

# 따라서, HUB에 Lane? 컨베이어? 인력? 을 더 두기로 결정

### 특히, 주문분산 처리 결과 병목현상은 거의 최적에 가까웠고 평균 처리 시간도 112.04초로 빨라졌지만, 총 주문 수가 1305개로 감소했기 때문에 시뮬레이션의 목적과 부합하지 않음 ★★★( 처리시간 빠르면서 총 주문 처리 수 많게)

# 입고 하프1, 피킹 하프1&풀1, HUB 하프1

In [ ]:
import simpy
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 시드 고정
random.seed(42)

# 작업 처리 시간 (초)
PROCESS_TIMES = {
    'IB': 20,
    'Picking': 60,
    'Packing': 10,
    'HUB': 20
}

# 시뮬레이션 설정
WORK_TIME = 9 * 60 * 60  # 9시간
ORDER_INTERVAL_STD_MORNING = 2
ORDER_INTERVAL_STD_AFTERNOON = 3

LUNCH_START = 2 * 60 * 60  # 11:00
LUNCH_END = 4 * 60 * 60    # 13:00

BREAK_PERIODS = [
    (5 * 60 * 60, 5 * 60 * 60 + 15 * 60),  # 14:00~14:15
    (7 * 60 * 60, 7 * 60 * 60 + 15 * 60),  # 16:00~16:15
]

HALF_TIME_START = 0
HALF_TIME_END = 4 * 60 * 60  # 13:00 (14400초)

class FulfillmentCenter:
    def __init__(self, env):
        self.env = env
        self.resources = {
            'IB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3)
            },
            'Picking': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3),
                4: simpy.Resource(env, capacity=4)
            },
            'Packing': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2)
            },
            'HUB': {
                1: simpy.Resource(env, capacity=1),
                2: simpy.Resource(env, capacity=2),
                3: simpy.Resource(env, capacity=3)  # 기본 2 + 오전 하프 1
            },
        }

    def is_reduced_capacity(self):
        now = self.env.now
        return LUNCH_START <= now <= LUNCH_END or any(start <= now <= end for start, end in BREAK_PERIODS)

    def get_resource(self, stage):
        now = self.env.now
        if stage == 'IB':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][3]
        elif stage == 'Picking':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][4]
            else:
                return self.resources[stage][3]
        elif stage == 'HUB':
            if HALF_TIME_START <= now < HALF_TIME_END:
                return self.resources[stage][3]  # 오전에만 3명
        return self.resources[stage][1] if self.is_reduced_capacity() else self.resources[stage][2]

    def process(self, stage):
        yield self.env.timeout(PROCESS_TIMES[stage])

def order_flow(env, fc, order_id, wait_times, gantt_log, log_trace):
    start = env.now
    log_trace.append(f"\n⏱️ [{int(env.now)}s] 주문 {order_id} ▶ 시작")
    for stage in ['IB', 'Picking', 'Packing', 'HUB']:
        with fc.get_resource(stage).request() as req:
            yield req
            stage_start = env.now
            log_trace.append(f"  🛠️ [{int(stage_start)}s] {stage:<8} ▶ 시작 (order {order_id})")
            yield env.process(fc.process(stage))
            stage_end = env.now
            duration = int(stage_end - stage_start)
            log_trace.append(f"  ✅ [{int(stage_end)}s] {stage:<8} ◀ 완료 (order {order_id}) - 소요: {duration}초")
            gantt_log.append((order_id, stage, stage_start, stage_end))
    total_time = env.now - start
    wait_times.append(total_time)
    log_trace.append(f"⏹️ [{int(env.now)}s] 주문 {order_id} ◀ 완료 - 총 소요 시간: {int(total_time)}초")

def generate_orders(env, fc, wait_times, gantt_log, log_trace):
    order_id = 0
    while env.now < WORK_TIME:
        interval = max(10, random.normalvariate(15, ORDER_INTERVAL_STD_MORNING)) if env.now < 4 * 60 * 60 else max(10, random.normalvariate(30, ORDER_INTERVAL_STD_AFTERNOON))
        env.process(order_flow(env, fc, order_id, wait_times, gantt_log, log_trace))
        yield env.timeout(interval)
        order_id += 1

def monitor_queues(env, fc, queue_log, interval=60):
    while env.now < WORK_TIME:
        for stage in ['IB', 'Picking', 'Packing', 'HUB']:
            queue_log[stage].append((env.now, len(fc.get_resource(stage).queue)))
        yield env.timeout(interval)

# 실행
env = simpy.Environment()
fc = FulfillmentCenter(env)
wait_times = []
queue_log = {stage: [] for stage in ['IB', 'Picking', 'Packing', 'HUB']}
gantt_log = []
log_trace = []

env.process(generate_orders(env, fc, wait_times, gantt_log, log_trace))
env.process(monitor_queues(env, fc, queue_log, interval=60))
env.run()


# 로그 출력
print("\n📋 시뮬레이션 상세 로그 (공정 흐름)")
for log in log_trace[10000:]:  # 앞 100줄만 출력
    print(log)


# 결과 출력 및 시각화
print(f"\n총 주문 수: {len(wait_times)}")


hub_done_count = 0
for log in log_trace:
    if 'HUB' in log and '◀ 완료' in log:
        # 로그에서 시간 추출
        try:
            time_part = log.split(']')[0]         # '✅ [39636s' → '[39636s'
            timestamp = int(time_part.strip('✅ [s'))  # 숫자만 추출
            if timestamp <= 32460:
                hub_done_count += 1
        except:
            continue

print(f"HUB 완료된 주문 수 (32460초 이내): {hub_done_count}")
print(f"평균 처리 시간: {sum(wait_times)/len(wait_times):.2f}초")

plt.figure(figsize=(10, 4))
plt.hist(wait_times, bins=30, edgecolor='black')
plt.title('Order Processing Time Distribution (HUB AM Half-Time Added)')
plt.xlabel('Total Time per Order (sec)')
plt.ylabel('Number of Orders')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
for stage in ['IB', 'Picking', 'Packing', 'HUB']:
    times, lengths = zip(*queue_log[stage])
    plt.plot(times, lengths, label=stage)
plt.title('Queue Length Over Time (per Process)')
plt.xlabel('Time (sec)')
plt.ylabel('Queue Length')
plt.ylim(0, 250)
plt.yticks(range(0, 251, 50))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

stage_colors = {'IB': 'skyblue', 'Picking': 'orange', 'Packing': 'lightgreen', 'HUB': 'salmon'}
plt.figure(figsize=(14, 8))
y_labels, y_ticks = [], []
order_limit = 50
for i, order_id in enumerate(sorted(set(o for o, _, _, _ in gantt_log))[:order_limit]):
    for stage, color in stage_colors.items():
        for log in gantt_log:
            if log[0] == order_id and log[1] == stage:
                start, duration = log[2], log[3] - log[2]
                plt.broken_barh([(start, duration)], (i * 10, 9), facecolors=color)
    y_labels.append(f"Order {order_id}")
    y_ticks.append(i * 10 + 4.5)
plt.yticks(y_ticks, y_labels)
plt.xlabel('Time (sec)')
plt.title('Gantt Chart: First 50 Orders')
plt.legend(handles=[mpatches.Patch(color=c, label=s) for s, c in stage_colors.items()])
plt.grid(True)
plt.tight_layout()
plt.show()


### 1. 그래프 1
▶ HUB 오전 하프 인력 투입으로 병목이 사라졌고,
병목으로 인한 지연 없이 빠르게 처리되어 처리 시간 분포가 압축됨

▶성과 매우 우수: 이전보다 평균 시간도 짧고, 분산도 낮음

### 2. 그래프 2
▶Picking - 전체 시뮬레이션 시간 동안 queue 길이 10 미만 유지
→ 과거에는 병목이었지만, 하프+풀 인력(=레인) 총 4명 추가 구성으로 해결

▶HUB - 이전에는 오전 7500~15000초에 병목 발생했으나, 이제는 완전히 제거